In [ ]:
import os
import pandas as pd
import duckdb
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()

TABLE = "era5"
CONTAINER = "rasterstats-public"
STAGE = "dev"
SAS = os.getenv("DSCI_AZ_BLOB_DEV_SAS")
ACCOUNT_NAME = "imb0chd0dev"

In [37]:
# Configure DuckDB with Azure credentials
conn = duckdb.connect()
conn.execute("INSTALL azure; LOAD azure;")
conn.execute(f"""
    CREATE SECRET azure_secret (
        TYPE AZURE,
        PROVIDER config,
        CONNECTION_STRING 'AccountName={ACCOUNT_NAME};SharedAccessSignature={SAS}'
    )
""")

In [42]:
conn.execute(f"""
    CREATE OR REPLACE VIEW {TABLE} AS
    SELECT * FROM read_parquet(
        'az://{CONTAINER}/{TABLE}/*/*.parquet',
        hive_partitioning = true
    )
""")

In [54]:
# Now query with plain SQL
df = conn.execute("""
    SELECT * FROM era5
    WHERE iso3 = 'NGA'
    AND pcode = 'NG001'
    ORDER BY valid_date
""").df()

In [55]:
df

,iso3,pcode,valid_date,adm_level,mean,median,min,max,count,sum,std
0,NGA,NG001,1981-01-01,1,1.064585,1.125336,0.301361,2.990723,154,163.94615,0.523139
1,NGA,NG001,1981-02-01,1,1.667791,1.184463,1.052856,3.511429,154,256.83975,0.772856
2,NGA,NG001,1981-03-01,1,4.975901,4.762650,2.891541,8.258820,154,766.28876,1.514842
3,NGA,NG001,1981-04-01,1,6.286472,5.943298,4.444122,9.439468,154,968.11676,1.437614
4,NGA,NG001,1981-05-01,1,10.377029,9.948730,8.779526,13.271332,154,1598.06250,1.300977
...,...,...,...,...,...,...,...,...,...,...,...
538,NGA,NG001,2025-11-01,1,3.618959,3.019333,1.426697,7.541657,154,557.31964,1.794280
539,NGA,NG001,2025-12-01,1,1.365581,1.093864,0.617027,3.092766,154,210.29950,0.678489
540,NGA,NG001,2026-01-01,1,1.045017,0.996590,0.337601,2.221108,154,160.93254,0.541554
541,NGA,NG001,2026-02-01,1,1.093146,0.877380,0.497818,2.159119,154,168.34450,0.445868


In [52]:
df_nga = stratus.load_parquet_from_blob(
    blob_name=f"{TABLE}/iso3=AFG/data.parquet",
    container_name=stratus.get_container_client(CONTAINER, "dev"),
)

HttpResponseError: The specified resource name length is not within the permissible limits.
RequestId:d423cc36-d01e-0045-6a6c-cec019000000
Time:2026-04-17T13:19:39.8378772Z
ErrorCode:OutOfRangeInput
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>OutOfRangeInput</Code><Message>The specified resource name length is not within the permissible limits.
RequestId:d423cc36-d01e-0045-6a6c-cec019000000
Time:2026-04-17T13:19:39.8378772Z</Message></Error>

In [48]:
df

,iso3,pcode,valid_date,adm_level,mean,median,min,max,count,sum,std,__index_level_0__
0,NGA,NG001,1981-01-01,1,1.064585,1.125336,0.301361,2.990723,154,163.94615,0.523139,3630700
1,NGA,NG001,1981-02-01,1,1.667791,1.184463,1.052856,3.511429,154,256.83975,0.772856,3635141
2,NGA,NG001,1981-03-01,1,4.975901,4.762650,2.891541,8.258820,154,766.28876,1.514842,3635178
3,NGA,NG001,1981-04-01,1,6.286472,5.943298,4.444122,9.439468,154,968.11676,1.437614,3635215
4,NGA,NG001,1981-05-01,1,10.377029,9.948730,8.779526,13.271332,154,1598.06250,1.300977,3635252
...,...,...,...,...,...,...,...,...,...,...,...,...
538,NGA,NG001,2025-11-01,1,3.618959,3.019333,1.426697,7.541657,154,557.31964,1.794280,4220134
539,NGA,NG001,2025-12-01,1,1.365581,1.093864,0.617027,3.092766,154,210.29950,0.678489,4227894
540,NGA,NG001,2026-01-01,1,1.045017,0.996590,0.337601,2.221108,154,160.93254,0.541554,4235652
541,NGA,NG001,2026-02-01,1,1.093146,0.877380,0.497818,2.159119,154,168.34450,0.445868,4243401


## 1. Read a single country — the primary access pattern

This is the fast path: reads only the Nigeria partition, ignores all other country files entirely.

In [ ]:
# Single country — DuckDB opens only NGA partition
df_nga = conn.execute("""
    SELECT * FROM era5
    WHERE iso3 = 'NGA'
    ORDER BY valid_date
""").df()

CatalogException: Catalog Error: Table with name era5 does not exist!
Did you mean "pg_description"?

LINE 2:     SELECT * FROM era5
                          ^

In [ ]:


print(f"Rows: {len(df_nga):,}")
print(f"Date range: {df_nga['issued_date'].min()} → {df_nga['issued_date'].max()}")
df_nga.head()


# %% [markdown]
# ## 2. Read multiple countries
#
# Load a few countries and concatenate — still only reads the
# relevant partition files.

# %%
countries = ["NGA", "ETH", "SDN", "SSD"]

df_multi = pd.concat([
    load_parquet_from_blob(
        container_client,
        blob_path=f"{TABLE}/iso3={iso3}/data.parquet",
    )
    for iso3 in countries
])

print(f"Countries: {sorted(df_multi['iso3'].unique())}")
print(f"Rows: {len(df_multi):,}")
df_multi.head()


# %% [markdown]
# ## 3. Query with DuckDB
#
# Once loaded into a DataFrame, DuckDB can query it efficiently
# in memory — fast filtering, aggregation, window functions.

# %%
conn = duckdb.connect()

# Full time series for one country, one leadtime
df_result = conn.execute("""
    SELECT
        valid_date,
        issued_date,
        leadtime,
        mean,
        min,
        max
    FROM df_nga
    WHERE leadtime = 7
    ORDER BY issued_date
""").df()

df_result.head(10)


# %% [markdown]
# ## 4. Aggregation example
#
# Compare mean forecast values across admin units for a specific issuance.

# %%
latest_issued = df_nga["issued_date"].max()

df_latest = conn.execute(f"""
    SELECT
        pcode,
        adm_level,
        valid_date,
        leadtime,
        ROUND(mean, 3) AS mean,
        ROUND(std, 3)  AS std,
        count
    FROM df_nga
    WHERE issued_date = '{latest_issued}'
    ORDER BY valid_date, leadtime
""").df()

print(f"Latest issuance: {latest_issued}")
print(f"Admin units: {df_latest['pcode'].nunique()}")
df_latest.head(10)


# %% [markdown]
# ## 5. Time series for a specific admin unit and leadtime

# %%
pcode = df_nga["pcode"].iloc[0]  # pick first pcode as example

df_ts = conn.execute(f"""
    SELECT
        issued_date,
        valid_date,
        leadtime,
        mean,
        min,
        max,
        std
    FROM df_nga
    WHERE pcode    = '{pcode}'
    AND   leadtime = 7
    ORDER BY issued_date
""").df()

print(f"Pcode: {pcode}")
print(f"Leadtime 7 issuances: {len(df_ts)}")
df_ts.head(10)


# %% [markdown]
# ## 6. Check metadata
#
# Quick sanity check on freshness and coverage before doing analysis.

# %%
import json

meta_blob = container_client.get_blob_client(f"{TABLE}/_metadata.json")
meta = json.loads(meta_blob.download_blob().readall())

print(f"Table:         {meta['table']}")
print(f"Last updated:  {meta['last_updated']}")
print(f"Total rows:    {meta['row_count']:,}")
print(f"Countries:     {meta['partition_count']}")
print(f"issued_date:   {meta['temporal_extent']['issued_date_min']} → {meta['temporal_extent']['issued_date_max']}")
print(f"valid_date:    {meta['temporal_extent']['valid_date_min']} → {meta['temporal_extent']['valid_date_max']}")